In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# A Logic Puzzle

The following exercise is taken from the book
<a href="https://www.amazon.de/Logeleien-Zweistein-ihren-Antworten-Wegner/dp/B006YF0VUE">"99 Logeleien von Zweistein"</a>.
This book has been published 1968.  It is written by
<a href="http://de.wikipedia.org/wiki/Thomas_von_Randow">Thomas von Randow</a>.

---
The gentlemen Amann, Bemann, Cemann and Demann are called - not necessarily in the same order - by their first names Erich, Fritz, Gustav and Heiner. They are all married to exactly one woman. We also know the following about them and their wives:

- Either Amann's first name is Heiner, or Bemann's wife is Inge.
- If Cemann is married to Josefa, then - **and only in this case** - Klara's husband is **not** called Fritz.
- If Josefa's husband is **not** called Erich, then Inge is married to Fritz.
- If Luise's husband is called Fritz, then Klara's husband's first name is **not** Gustav.
- If the wife of Fritz is called Inge, then Erich is **not** married to Josefa.
- If Fritz is **not** married to Luise, then Gustav's wife's name is Klara.
- Either Demann is married to Luise, or Cemann is called Gustav.

*What are the full names of these gentlemen, and what are their wives' first names?*

---

We are going to solve this problem by coding it in propositional logic and we will solve the resulting set of clauses using the Davis-Putnam algorithm.  In order to code the problem, we will use the following propositional variables:

- $\texttt{MaleName<}x\texttt{,}z\texttt{>}$ for any male first name $x$ and any surname $z$ expresses
  that the gentleman with first name $x$ has surname $z$.
- $\texttt{Married<}x\texttt{,}y\texttt{>}$ for any male first name $x$ and any female first name $y$ expresses
  that the gentleman with first name $x$ is married to the lady with first name $y$.
- $\texttt{FemaleName<}x\texttt{,}z\texttt{>}$ for any female first name $x$ and any last name $z$ expresses
  that the lady with first name $x$ has surname $z$.

We are using the symbols $\texttt{<}$ and $\texttt{>}$ as part of the propositional variables because we want to show the structure of these variables and the parser for propositional logic accepts these symbols as part of propositional variables.

The notebook `06-Davis-Putnam.ipynb` provides the function `solve`.  It imports the notebook `04-CNF.ipynb`, which defines the types `literal`, `clause`, and `cnf`.  The module `RSet` is loaded by `setup.ml`.

In [ ]:
import_notebook "06-Davis-Putnam.ipynb"

In [ ]:
let first_male   = RSet.of_list ["Erich"; "Fritz"; "Gustav"; "Heiner"]
let first_female = RSet.of_list ["Inge"; "Josefa"; "Klara"; "Luise"]
let sur_names    = RSet.of_list ["Amann"; "Bemann"; "Cemann"; "Demann"]

Instead of using the parser for propositional logic, we will create the clauses directly.  A negated variable `p` is written as `Neg p` and the function `complement` from the notebook `06-Davis-Putnam.ipynb` computes the complement of a literal.  The TypeScript version defines a function `not` that negates a variable.  In OCaml, we can not use this name, since `not` is the negation of Boolean values.

The function `make_var` takes three arguments:
- `f` is the name of a relation,
- `x` is a name,
- `y` is a name.

It returns a propositional variable of the form `f<x,y>`.

In [ ]:
let make_var (f : string) (x : string) (y : string) : string =
  Printf.sprintf "%s<%s,%s>" f x y

In [ ]:
make_var "Married" "Heiner" "Klara"

The function `at_most_one` takes one argument:
- `s` is a set of propositional variables.

It returns a set of clauses expressing the fact that at most one of the variables of `s` is `true`.

In [ ]:
let at_most_one (s : string RSet.t) : cnf =
  RSet.flat_map
    (fun p -> RSet.filter_map (fun q -> if p <> q then Some (RSet.of_list [Neg p; Neg q]) else None) s)
    s

The function `at_least_one` takes one argument:
- `s` is a set of propositional variables.

It returns a set of clauses expressing the fact that at least one of the variables of `s` is `true`.

In [ ]:
let at_least_one (s : string RSet.t) : cnf =
  RSet.singleton (RSet.map (fun p -> Pos p) s)

The function `exactly_one` takes one argument:
- `s` is a set of propositional variables.

It returns a set of clauses expressing the fact that exactly one of the variables in the set `s` is true.

In [ ]:
let exactly_one (s : string RSet.t) : cnf =
  RSet.union (at_most_one s) (at_least_one s)

In [ ]:
exactly_one (RSet.of_list ["a"; "b"; "c"])

The function `bijective` takes three arguments:
- `a` is a set of names,
- `b` is a set of names that has the same number of elements as `a`,
- `f` is the name of a relation.

It computes a set of clauses that is equivalent to the formula
$$   \bigl(\forall x \in A: \exists! y \in B: f\langle x, y\rangle\bigr) \wedge
     \bigl(\forall y \in B: \exists! x \in A: f\langle x, y\rangle\bigr)
$$
Here the expression $f\langle x,y\rangle$ is the name of a propositional variable and the expression $\exists!x:p(x)$ is to be read as "There exists exactly one $x$ such that $p(x)$ holds".

Replace every occurrence of `failwith "your code here"` with your code.  The function `failwith` raises an exception.  As it can be used in place of an expression of any type, it serves as a placeholder.

In [ ]:
let bijective (a : string RSet.t) (b : string RSet.t) (f : string) : cnf =
  failwith "your code here"

In [ ]:
bijective (RSet.of_list ["a"; "b"]) (RSet.of_list ["x"; "y"]) "f"

We introduce the logical connectives `implies`, `exclusive_or`, `equivalence`, and `implies_and` as functions that directly yield sets of clauses.  Each of these functions takes literals as arguments and returns the set of clauses of the corresponding formula:
- `implies a b` represents $a \rightarrow b$, which is equivalent to $\overline{a} \vee b$,
- `exclusive_or a b` represents $a \leftrightarrow \neg b$, which is equivalent to $(a \vee b) \wedge (\overline{a} \vee \overline{b})$,
- `equivalence a b` represents $a \leftrightarrow b$, which is equivalent to $(\overline{a} \vee b) \wedge (a \vee \overline{b})$,
- `implies_and a b c` represents $a \wedge b \rightarrow c$, which is equivalent to $\overline{a} \vee \overline{b} \vee c$.

In [ ]:
let implies (a : literal) (b : literal) : cnf =
  RSet.of_list [RSet.of_list [complement a; b]]

let exclusive_or (a : literal) (b : literal) : cnf =
  RSet.of_list [RSet.of_list [a; b]; RSet.of_list [complement a; complement b]]

let equivalence (a : literal) (b : literal) : cnf =
  RSet.of_list [RSet.of_list [complement a; b]; RSet.of_list [a; complement b]]

let implies_and (a : literal) (b : literal) (c : literal) : cnf =
  RSet.of_list [RSet.of_list [complement a; complement b; c]]

In [ ]:
exclusive_or (Pos "p") (Pos "q")

The function `consistent_names` takes three arguments:
- `first_male` is the set of male first names,
- `first_female` is the set of female first names,
- `sur_names` is the set of surnames.

It returns a set of clauses that ensures that if the man `x` has the surname `z` and the woman `y` also has the surname `z`, then `x` and `y` have to be married.

In [ ]:
let consistent_names (first_male : string RSet.t) (first_female : string RSet.t)
                     (sur_names : string RSet.t) : cnf =
  RSet.flat_map (fun x ->
    RSet.flat_map (fun y ->
      RSet.flat_map (fun z ->
        let a = make_var "MaleName"   x z in
        let b = make_var "FemaleName" y z in
        let c = make_var "Married"    x y in
        implies_and (Pos a) (Pos b) (Pos c))
        sur_names)
      first_female)
    first_male

In [ ]:
consistent_names first_male first_female sur_names

Now we translate the constraints of the puzzle into clauses.  The function `compute_clauses` takes the same three arguments as `consistent_names` and returns the set of all clauses describing the puzzle.

In [ ]:
let compute_clauses (first_male : string RSet.t) (first_female : string RSet.t)
                    (sur_names : string RSet.t) : cnf =
  RSet.unions [
    (* Jedem männlichen Vornamen ist genau ein Nachname zugeordnet und umgekehrt. *)
    failwith "your code here";
    (* Jeder Mann ist mit genau einer Frau verheiratet und umgekehrt. *)
    failwith "your code here";
    (* Jede Frau hat genau einen Nachnamen. *)
    failwith "your code here";
    (* Die Namen sind konsistent. *)
    failwith "your code here";
    (* Entweder ist Amanns Vorname Heiner, oder Bemanns Frau heisst Inge. *)
    failwith "your code here";
    (* Wenn Cemann mit Josefa verheiratet ist, dann – und nur in diesem Falle – heisst Klaras Mann nicht Fritz. *)
    failwith "your code here";
    (* Wenn Josefas Mann nicht Erich heisst, dann ist Inge mit Fritz verheiratet. *)
    failwith "your code here";
    (* Wenn Luises Mann Fritz heisst, dann ist der Vorname von Klaras Mann nicht Gustav. *)
    failwith "your code here";
    (* Wenn die Frau von Fritz Inge heisst, dann ist Erich nicht mit Josefa verheiratet. *)
    failwith "your code here";
    (* Wenn Fritz nicht mit Luise verheiratet ist, dann heisst Gustavs Frau Klara. *)
    failwith "your code here";
    (* Entweder ist Demann mit Luise verheiratet, oder Cemann heisst Gustav. *)
    failwith "your code here"
  ]

In [ ]:
let clauses = compute_clauses first_male first_female sur_names

There are 242 different clauses.

In [ ]:
RSet.cardinal clauses

The function `compute_solution` takes the same three arguments as `compute_clauses`.  It computes the clauses describing the puzzle and solves them.

In [ ]:
let compute_solution (first_male : string RSet.t) (first_female : string RSet.t)
                     (sur_names : string RSet.t) : cnf =
  solve (compute_clauses first_male first_female sur_names)

In [ ]:
let solution = compute_solution first_male first_female sur_names

The function `only_positive` takes one argument:
- `solution` is a set of unit clauses.

It returns the set of the variables of those unit clauses that contain a positive literal.  The function `arb` from the notebook `06-Davis-Putnam.ipynb` returns the only element of a unit clause.

In [ ]:
let only_positive (solution : cnf) : string RSet.t =
  RSet.filter_map (fun c -> match arb c with Pos p -> Some p | Neg _ -> None) solution

In [ ]:
only_positive solution

The function `extract_args` takes one argument:
- `s` is a variable of the form `f<x,y>`.

It returns the pair `(x, y)`.  The format `"%[^<]<%[^,],%[^>]>"` of `Scanf.sscanf` reads the characters up to the first `<`, the characters up to the next `,`, and the characters up to the next `>`.  The first of these three strings is ignored.

In [ ]:
let extract_args (s : string) : string * string =
  Scanf.sscanf s "%[^<]<%[^,],%[^>]>" (fun _ x y -> (x, y))

In [ ]:
extract_args "Married<Fritz,Inge>"

The function `display_solution` takes one argument:
- `solution` is a set of unit clauses representing a solution of the puzzle.

It prints the full name of every gentleman together with the first name of his wife.

**Implementation:** The association list `married` maps every male first name to the first name of his wife, while `names` maps every male first name to his surname.

In [ ]:
let display_solution (solution : cnf) : unit =
  let positive = RSet.to_list (only_positive solution) in
  let pairs prefix =
    List.filter_map
      (fun v -> if String.starts_with ~prefix v then Some (extract_args v) else None)
      positive
  in
  let married = pairs "Married" in
  let names   = pairs "MaleName" in
  List.iter
    (fun (x, wife) -> Printf.printf "%s %s is married to %s.\n" x (List.assoc x names) wife)
    married

In [ ]:
display_solution solution

## Checking the Uniqueness of the Solution

The function `negate_solution` takes one argument:
- `unit_clauses` is a set of unit clauses.

It returns a clause that is the negation of `unit_clauses`.

In [ ]:
let negate_solution (unit_clauses : cnf) : clause =
  RSet.map (fun u -> complement (arb u)) unit_clauses

In [ ]:
negate_solution (RSet.of_list [RSet.of_list [Pos "a"]; RSet.of_list [Neg "b"]])

The function `check_uniqueness` takes two arguments:
- `solution` is a set of unit clauses representing a solution of the puzzle,
- `base_clauses` is the set of clauses describing the puzzle.

It adds the negation of `solution` to `base_clauses`.  If the resulting set of clauses is inconsistent, i.e. if `solve` returns the set $\bigl\{\{\}\bigr\}$, the solution is unique.

In [ ]:
let check_uniqueness (solution : cnf) (base_clauses : cnf) : unit =
  let augmented_clauses = RSet.add (negate_solution solution) base_clauses in
  let alternative = solve augmented_clauses in
  if alternative = RSet.singleton RSet.empty then
    print_endline "Well done: The solution is unique!"
  else
    print_endline "ERROR: The solution is not unique!"

In [ ]:
check_uniqueness solution clauses